In [1]:

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
customer_id = df['customerID']
data = df.drop(['customerID'], axis=1)

cols = ['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# data['gender'] = data['gender'].map({'Male': 1, 'Female': 0})
# cols_without_yesno = []

# print(df['Partner'].value_counts().index.tolist())    # check output, can be misarranged

# for col in cols:
    
#     if df[col].dtype == 'object':
#         # print(df[col].value_counts())
#         pass

#     if df[col].value_counts().index.tolist() != ['No', 'Yes']:
#         cols_without_yesno.append(col)

#     if df[col].value_counts().index.tolist() == ['No', 'Yes']:
#         df[col] = df[col].map({'Yes': 1, 'No': 0})

data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

data.sample(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4860,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,No
4750,Male,0,No,No,9,No,No phone service,DSL,Yes,Yes,Yes,No,No,No,One year,No,Mailed check,39.55,373.00,No
6634,Female,0,Yes,Yes,10,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Credit card (automatic),102.10,1068.85,Yes
202,Male,0,Yes,Yes,71,Yes,Yes,Fiber optic,Yes,No,Yes,No,Yes,Yes,Two year,No,Electronic check,105.55,7405.50,No
5705,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.65,19.65,No


In [2]:
cols_without_yesno = []
cols_with_yesno = ['gender',]

for col in cols:
    # print(data[col].nunique())
    if data[col].nunique() > 2:
        cols_without_yesno.append(col)
    else :
        cols_with_yesno.append(col)
print(len(cols_without_yesno))
print(len(cols_with_yesno))
print(len(numeric_cols))

10
6
3


In [3]:
from sklearn.model_selection import train_test_split

x = data.drop(['Churn'], axis=1)
y = data['Churn'].map({'Yes': 1, 'No': 0})
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42, test_size=0.2)

In [4]:
x_train.sample(5)
y_train.sample(8)

2737    0
3354    0
5923    0
4284    0
2420    0
6840    0
3321    0
3852    0
Name: Churn, dtype: int64

### Transformers

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# trf1 = ColumnTransformer(transformers=[
#     ('impute_totalcharges', SimpleImputer(), [18])
# ], remainder='passthrough')

# trf2 = ColumnTransformer(transformers=[
#     ("mapping", OrdinalEncoder(), [0])
# ], remainder='passthrough')

# trf3 = ColumnTransformer(transformers=[
#     ("ohe", OneHotEncoder(handle_unknown="ignore"), [6, 7, 8, 9, 10, 11, 12, 13, 14, 16])
# ], remainder='passthrough')

# trf4 = SelectKBest(k=8)

# trf5 = RandomForestClassifier()

processor = ColumnTransformer(transformers=[
    ("Gender_encoding", OrdinalEncoder(), cols_with_yesno),
    ("Missing_values", SimpleImputer(), ['TotalCharges']),
    ("OHE", OneHotEncoder(), cols_without_yesno)
], remainder='passthrough')


In [6]:
from sklearn.pipeline import make_pipeline, Pipeline

pipe_1 = Pipeline([
    # ('trf1',trf1),
    # ('trf2',trf2),
    # ('trf3',trf3),
    # ('trf4',trf4),
    # ('trf5',trf5)
    ('processor', processor),
    ('select_features', SelectKBest(k=8)),
    ('modeling', RandomForestClassifier())
])

# pipe = make_pipeline(trf1, trf2, trf3, trf4, trf5)
pipe_1.fit(x_train, y_train)

,steps,"[('processor', ...), ('select_features', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Gender_encoding', ...), ('Missing_values', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [7]:
pipe_1.get_params()

{'memory': None,
 'steps': [('processor',
   ColumnTransformer(remainder='passthrough',
                     transformers=[('Gender_encoding', OrdinalEncoder(),
                                    ['gender', 'SeniorCitizen', 'Partner',
                                     'Dependents', 'PhoneService',
                                     'PaperlessBilling']),
                                   ('Missing_values', SimpleImputer(),
                                    ['TotalCharges']),
                                   ('OHE', OneHotEncoder(),
                                    ['MultipleLines', 'InternetService',
                                     'OnlineSecurity', 'OnlineBackup',
                                     'DeviceProtection', 'TechSupport',
                                     'StreamingTV', 'StreamingMovies', 'Contract',
                                     'PaymentMethod'])])),
  ('select_features', SelectKBest(k=8)),
  ('modeling', RandomForestClassifier())],
 'transfor

In [8]:
from sklearn.metrics import accuracy_score, roc_auc_score
y_pred = pipe_1.predict(x_test)
print(f"accuracy {accuracy_score(y_test, y_pred)}")
print(f"AUC ROC {roc_auc_score(y_test, y_pred)}")

accuracy 0.7601135557132718
AUC ROC 0.6978997381142154


### XGB

In [9]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb = XGBClassifier()
params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_split_loss': [0, 0.5, 1],
    'max_depth': [3, 6, 10],
}

processor_xgb = ColumnTransformer(transformers=[
    ("Gender_encoding", OrdinalEncoder(), cols_with_yesno),
    ("Missing_values", SimpleImputer(), ['TotalCharges']),
    ("OHE", OneHotEncoder(), cols_without_yesno)
], remainder='passthrough')

grid = GridSearchCV(
    estimator=xgb,
    param_grid=params, 
    scoring='roc_auc',
    n_jobs=-1,
    cv=5,
)

pipe_2 = make_pipeline(processor_xgb, grid)
pipe_2.fit(x_train, y_train)

,steps,"[('columntransformer', ...), ('gridsearchcv', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Gender_encoding', ...), ('Missing_values', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
y_pred = pipe_2.predict(x_test)
y_pred_proba = pipe_2.predict_proba(x_test)[:, 1]
print(f"accuracy {accuracy_score(y_test, y_pred)}")
print(f"AUC ROC {roc_auc_score(y_test, y_pred_proba)}")

accuracy 0.8112136266855926
AUC ROC 0.8631866738435103


In [11]:
from catboost import CatBoostClassifier

pipe_3 = make_pipeline(processor, CatBoostClassifier(verbose=0))
pipe_3.fit(x_train, y_train)
y_pred = pipe_3.predict(x_test)
y_proba = pipe_3.predict_proba(x_test)[:, 1]

print(f"accuracy {accuracy_score(y_test, y_pred)}")
print(f"AUC ROC {roc_auc_score(y_test, y_proba)}")

accuracy 0.8055358410220014
AUC ROC 0.8573175339261132
